In [1]:
import os
import re
import random
from dataclasses import dataclass
from collections import Counter

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import mean_absolute_error

In [2]:
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)

In [3]:
# =========================
# RNN / LSTM BASELINE CONFIG - STRONGER VERSION
# Essay embedding only
# No prompt, no discourse features
# =========================

RNN_OUTPUT_DIR = "/content/ielts_lstm_essay_only_trait_stacks"
os.makedirs(RNN_OUTPUT_DIR, exist_ok=True)

TRAIT_COLS = ["TR", "CC", "LR", "GRA"]

MIN_BAND = 0.0
MAX_BAND = 9.0

# =========================
# Text / vocab config
# =========================

MAX_VOCAB_SIZE = 50000
MAX_LEN = 900
MIN_FREQ = 2

# =========================
# Model config
# =========================

EMBED_DIM = 300
RNN_HIDDEN = 384
RNN_LAYERS = 2
BIDIRECTIONAL = True

STACK_HIDDEN = 512
DROPOUT = 0.35

# =========================
# Training config
# =========================

BATCH_SIZE = 32
EPOCHS = 80

LR = 5e-4
WEIGHT_DECAY = 5e-5
MAX_GRAD_NORM = 1.0

# =========================
# Early stopping config
# =========================

EARLY_STOPPING_PATIENCE = 8
MIN_DELTA = 1e-4
epochs_no_improve = 0

# =========================
# Overall consistency loss
# Keep False for clean multi-trait baseline
# =========================

USE_OPTIONAL_OVERALL_CONSISTENCY_LOSS = False
OVERALL_CONSISTENCY_WEIGHT = 0.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

DEVICE: cuda


In [4]:
TRAIN_PATH = "/content/train.csv"
VAL_PATH   = "/content/val.csv"
TEST_PATH  = "/content/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

display(train_df.head())

Train: (7743, 17)
Val: (969, 17)
Test: (970, 17)

Train columns:
['prompt', 'essay', 'evaluation', 'essay_len', 'TR', 'CC', 'LR', 'GRA', 'n_found', 'parse_quality', 'overall_raw', 'band', 'prompt_relevance', 'lexical_diversity', 'readability_score', 'target_text', 'source']


,prompt,essay,evaluation,essay_len,TR,CC,LR,GRA,n_found,parse_quality,overall_raw,band,prompt_relevance,lexical_diversity,readability_score,target_text,source
0,Some employers believe that job applicants’ so...,There has been much discussion revolving aroun...,Task Achievement:\r\n\r\n- The candidate has e...,273,7.5,7.0,7.0,7.0,4,good,7.125,7.0,0.592603,0.581818,40.754804,Analysis: This essay has a lexical diversity o...,hf
1,Some people believe that teenagers should be r...,A nation is known as a vast garden and childre...,Task Achievement:\r\nThe essay effectively add...,324,4.0,4.0,3.0,3.0,4,good,3.500,3.5,0.600165,0.563077,35.677525,Analysis: This essay has a lexical diversity o...,hf
2,some people say that economic growth is the on...,"In the modern world ,a burning issue arises th...",Task Achievement:\r\n\r\nThe essay adequately ...,348,5.5,5.0,5.0,5.0,4,good,5.125,5.0,0.746589,0.452450,40.414844,Analysis: This essay has a lexical diversity o...,hf
3,Some believe that people should make efforts t...,The issue of climate change was always debatab...,Task Achievement:\r\n- The candidate has adequ...,307,7.0,7.0,6.0,6.0,4,good,6.500,6.5,0.628859,0.629870,39.378580,Analysis: This essay has a lexical diversity o...,hf
4,Nowadays families move to different countries ...,Immigrating to other nations for business purp...,Task Achievement:\r\nThe candidate has effecti...,311,7.5,8.0,7.0,7.5,4,good,7.500,7.5,0.586910,0.612179,40.385892,Analysis: This essay has a lexical diversity o...,hf


In [5]:
required_cols = ["essay", "TR", "CC", "LR", "GRA"]

for name, df_ in [
    ("train_df", train_df),
    ("val_df", val_df),
    ("test_df", test_df),
]:
    missing = [c for c in required_cols if c not in df_.columns]

    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")

    df_["essay"] = df_["essay"].fillna("").astype(str)

    for col in ["TR", "CC", "LR", "GRA"]:
        df_[col] = df_[col].astype(float)

    if "Overall" in df_.columns:
        df_["Overall"] = df_["Overall"].astype(float)

print("All datasets are ready.")

All datasets are ready.


In [6]:
# =========================
# METRIC FUNCTIONS
# For essay-only RNN/LSTM baseline
# Includes derived Overall QWK
# =========================

from sklearn.metrics import cohen_kappa_score, mean_absolute_error

def band_to_scaled(x):
    """
    Convert IELTS band score from [0, 9] to scaled score [0, 1].
    """
    x = np.asarray(x, dtype=np.float32)
    return (x - MIN_BAND) / (MAX_BAND - MIN_BAND)


def scaled_to_band(x):
    """
    Convert scaled score [0, 1] back to IELTS band score [0, 9],
    then round to nearest 0.5 band.
    """
    x = np.asarray(x, dtype=np.float32)

    band = x * (MAX_BAND - MIN_BAND) + MIN_BAND
    band = np.clip(band, MIN_BAND, MAX_BAND)

    # Round to nearest IELTS half-band
    band = np.round(band * 2) / 2

    return band


def qwk_band(y_true, y_pred):
    """
    Compute Quadratic Weighted Kappa for IELTS band scores.
    IELTS half-band scores are multiplied by 2 and converted to integers.
    Example: 6.5 -> 13, 7.0 -> 14
    """
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)

    valid = ~np.isnan(y_true) & ~np.isnan(y_pred)

    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if len(y_true) == 0:
        return np.nan

    y_true_int = (y_true * 2).astype(int)
    y_pred_int = (y_pred * 2).astype(int)

    return cohen_kappa_score(
        y_true_int,
        y_pred_int,
        weights="quadratic",
    )


def compute_metrics(y_true_scaled, y_pred_scaled, overall_true=None):
    """
    Compute trait-level QWK/MAE and derived overall QWK/MAE.

    y_true_scaled: shape [N, 4], gold TR/CC/LR/GRA in scaled form [0, 1]
    y_pred_scaled: shape [N, 4], predicted TR/CC/LR/GRA in scaled form [0, 1]
    overall_true: shape [N], gold Overall band, optional

    Derived Overall prediction is computed as:
        mean(pred_TR, pred_CC, pred_LR, pred_GRA)
    then rounded to nearest 0.5 band.
    """
    y_true_band = scaled_to_band(y_true_scaled)
    y_pred_band = scaled_to_band(y_pred_scaled)

    metrics = {}

    trait_qwks = []
    trait_maes = []

    # =========================
    # Trait metrics
    # =========================

    for i, trait in enumerate(TRAIT_COLS):
        qwk = qwk_band(
            y_true_band[:, i],
            y_pred_band[:, i],
        )

        mae = mean_absolute_error(
            y_true_band[:, i],
            y_pred_band[:, i],
        )

        metrics[f"{trait}_qwk"] = qwk
        metrics[f"{trait}_mae"] = mae

        trait_qwks.append(qwk)
        trait_maes.append(mae)

    metrics["mean_trait_qwk"] = float(np.nanmean(trait_qwks))
    metrics["mean_trait_mae"] = float(np.nanmean(trait_maes))

    # =========================
    # Derived Overall prediction
    # =========================

    derived_overall_pred = np.mean(y_pred_band, axis=1)
    derived_overall_pred = np.round(derived_overall_pred * 2) / 2
    derived_overall_pred = np.clip(
        derived_overall_pred,
        MIN_BAND,
        MAX_BAND,
    )

    # Optional: also compute derived gold overall from gold traits
    derived_overall_gold_from_traits = np.mean(y_true_band, axis=1)
    derived_overall_gold_from_traits = np.round(derived_overall_gold_from_traits * 2) / 2
    derived_overall_gold_from_traits = np.clip(
        derived_overall_gold_from_traits,
        MIN_BAND,
        MAX_BAND,
    )

    metrics["derived_overall_from_traits_qwk"] = qwk_band(
        derived_overall_gold_from_traits,
        derived_overall_pred,
    )

    metrics["derived_overall_from_traits_mae"] = mean_absolute_error(
        derived_overall_gold_from_traits,
        derived_overall_pred,
    )

    # =========================
    # Compare derived Overall with real Overall column if available
    # =========================

    if overall_true is not None:
        overall_true = np.asarray(overall_true, dtype=np.float32)

        valid = ~np.isnan(overall_true)

        if valid.any():
            metrics["derived_overall_qwk"] = qwk_band(
                overall_true[valid],
                derived_overall_pred[valid],
            )

            metrics["derived_overall_mae"] = mean_absolute_error(
                overall_true[valid],
                derived_overall_pred[valid],
            )

    return metrics, y_true_band, y_pred_band, derived_overall_pred

In [7]:
def simple_tokenize(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9.,!?;:'\"()\- ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()

class Vocab:
    def __init__(self, max_size=30000, min_freq=2):
        self.max_size = max_size
        self.min_freq = min_freq

        self.pad_token = "<PAD>"
        self.unk_token = "<UNK>"

        self.stoi = {
            self.pad_token: 0,
            self.unk_token: 1,
        }
        self.itos = [self.pad_token, self.unk_token]

    def fit(self, texts):
        counter = Counter()
        for text in texts:
            counter.update(simple_tokenize(text))

        words = [
            word for word, freq in counter.most_common()
            if freq >= self.min_freq
        ]

        words = words[: self.max_size - len(self.itos)]

        for word in words:
            if word not in self.stoi:
                self.stoi[word] = len(self.itos)
                self.itos.append(word)

        print(f"Vocab size: {len(self.itos)}")

    def encode(self, text, max_len):
        tokens = simple_tokenize(text)
        ids = [self.stoi.get(tok, self.stoi[self.unk_token]) for tok in tokens]

        if len(ids) > max_len:
            ids = ids[:max_len]

        if len(ids) == 0:
            ids = [self.stoi[self.unk_token]]

        return ids
vocab = Vocab(max_size=MAX_VOCAB_SIZE, min_freq=MIN_FREQ)
vocab.fit(train_df["essay"].astype(str).tolist())

Vocab size: 27362


In [8]:
class IELTSEssayOnlyDataset(Dataset):
    def __init__(self, df, vocab, max_len=600):
        self.df = df.reset_index(drop=True)
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        input_ids = self.vocab.encode(row["essay"], self.max_len)

        labels_band = row[TRAIT_COLS].astype(float).values.astype(np.float32)
        labels_scaled = band_to_scaled(labels_band).astype(np.float32)

        if "Overall" in self.df.columns and not pd.isna(row.get("Overall", np.nan)):
            overall_band = np.float32(row["Overall"])
        else:
            overall_band = np.float32(np.nan)

        return {
            "input_ids": input_ids,
            "labels": labels_scaled,
            "overall_band": overall_band,
        }

@dataclass
class RNNDataCollator:
    pad_id: int = 0

    def __call__(self, batch):
        lengths = [len(x["input_ids"]) for x in batch]
        max_len = max(lengths)

        input_ids = []
        attention_mask = []

        for item in batch:
            ids = item["input_ids"]
            pad_len = max_len - len(ids)

            input_ids.append(ids + [self.pad_id] * pad_len)
            attention_mask.append([1] * len(ids) + [0] * pad_len)

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "lengths": torch.tensor(lengths, dtype=torch.long),
            "labels": torch.tensor(np.stack([x["labels"] for x in batch]), dtype=torch.float32),
            "overall_band": torch.tensor([x["overall_band"] for x in batch], dtype=torch.float32),
        }

train_dataset = IELTSEssayOnlyDataset(train_df, vocab, max_len=MAX_LEN)
val_dataset   = IELTSEssayOnlyDataset(val_df, vocab, max_len=MAX_LEN)
test_dataset  = IELTSEssayOnlyDataset(test_df, vocab, max_len=MAX_LEN)

collator = RNNDataCollator(pad_id=vocab.stoi["<PAD>"])

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
)

In [9]:
class TraitStack(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, dropout=0.3):
        super().__init__()

        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),

            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim // 4, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)

class EssayLSTMTraitScorer(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=200,
        rnn_hidden=256,
        rnn_layers=2,
        bidirectional=True,
        stack_hidden=256,
        dropout=0.3,
        pad_id=0,
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=pad_id,
        )

        self.embedding_dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=rnn_hidden,
            num_layers=rnn_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if rnn_layers > 1 else 0.0,
        )

        direction_factor = 2 if bidirectional else 1
        encoder_dim = rnn_hidden * direction_factor

        # Pooling: mean + max + last hidden
        representation_dim = encoder_dim * 3

        self.shared_norm = nn.LayerNorm(representation_dim)

        self.stacks = nn.ModuleDict({
            trait: TraitStack(
                input_dim=representation_dim,
                hidden_dim=stack_hidden,
                dropout=dropout,
            )
            for trait in TRAIT_COLS
        })

    def masked_mean_pooling(self, outputs, mask):
        mask = mask.unsqueeze(-1).float()
        summed = (outputs * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1.0)
        return summed / counts

    def masked_max_pooling(self, outputs, mask):
        mask = mask.unsqueeze(-1).bool()
        outputs = outputs.masked_fill(~mask, -1e9)
        return outputs.max(dim=1).values

    def forward(self, input_ids, attention_mask, lengths):
        embedded = self.embedding(input_ids)
        embedded = self.embedding_dropout(embedded)

        lengths_cpu = lengths.detach().cpu()

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths_cpu,
            batch_first=True,
            enforce_sorted=False,
        )

        packed_outputs, (h_n, c_n) = self.lstm(packed)

        outputs, _ = nn.utils.rnn.pad_packed_sequence(
            packed_outputs,
            batch_first=True,
            total_length=input_ids.size(1),
        )

        mean_pool = self.masked_mean_pooling(outputs, attention_mask)
        max_pool = self.masked_max_pooling(outputs, attention_mask)

        if self.lstm.bidirectional:
            last_forward = h_n[-2]
            last_backward = h_n[-1]
            last_hidden = torch.cat([last_forward, last_backward], dim=-1)
        else:
            last_hidden = h_n[-1]

        essay_repr = torch.cat([mean_pool, max_pool, last_hidden], dim=-1)
        essay_repr = self.shared_norm(essay_repr)

        preds = []
        for trait in TRAIT_COLS:
            pred = self.stacks[trait](essay_repr)
            preds.append(pred)

        preds = torch.stack(preds, dim=1)

        # giữ output trong range scaled 0-1
        preds = torch.sigmoid(preds)

        return preds

model = EssayLSTMTraitScorer(
    vocab_size=len(vocab.itos),
    embed_dim=EMBED_DIM,
    rnn_hidden=RNN_HIDDEN,
    rnn_layers=RNN_LAYERS,
    bidirectional=BIDIRECTIONAL,
    stack_hidden=STACK_HIDDEN,
    dropout=DROPOUT,
    pad_id=vocab.stoi["<PAD>"],
).to(DEVICE)

print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

EssayLSTMTraitScorer(
  (embedding): Embedding(27362, 300, padding_idx=0)
  (embedding_dropout): Dropout(p=0.35, inplace=False)
  (lstm): LSTM(300, 384, num_layers=2, batch_first=True, dropout=0.35, bidirectional=True)
  (shared_norm): LayerNorm((2304,), eps=1e-05, elementwise_affine=True)
  (stacks): ModuleDict(
    (TR): TraitStack(
      (net): Sequential(
        (0): LayerNorm((2304,), eps=1e-05, elementwise_affine=True)
        (1): Linear(in_features=2304, out_features=512, bias=True)
        (2): GELU(approximate='none')
        (3): Dropout(p=0.35, inplace=False)
        (4): Linear(in_features=512, out_features=256, bias=True)
        (5): GELU(approximate='none')
        (6): Dropout(p=0.35, inplace=False)
        (7): Linear(in_features=256, out_features=128, bias=True)
        (8): GELU(approximate='none')
        (9): Dropout(p=0.35, inplace=False)
        (10): Linear(in_features=128, out_features=1, bias=True)
      )
    )
    (CC): TraitStack(
      (net): Sequential(

In [10]:
def criterion_loss_lstm(pred_scaled, labels_scaled, overall_band=None):
    loss = 0.0

    for i, trait in enumerate(TRAIT_COLS):
        loss = loss + F.mse_loss(pred_scaled[:, i], labels_scaled[:, i])

    loss = loss / len(TRAIT_COLS)

    if USE_OPTIONAL_OVERALL_CONSISTENCY_LOSS and overall_band is not None:
        valid = ~torch.isnan(overall_band)

        if valid.any():
            pred_overall_scaled = pred_scaled.mean(dim=1)

            gold_overall_scaled = (
                overall_band.to(pred_scaled.device) - MIN_BAND
            ) / (MAX_BAND - MIN_BAND)

            loss = loss + OVERALL_CONSISTENCY_WEIGHT * F.mse_loss(
                pred_overall_scaled[valid],
                gold_overall_scaled[valid],
            )

    return loss

In [11]:
from tqdm.auto import tqdm
from sklearn.metrics import mean_absolute_error

def move_batch_to_device(batch):
    out = {}

    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            out[k] = v.to(DEVICE)
        else:
            out[k] = v

    return out

@torch.no_grad()
def predict_loader_lstm(model, loader):
    model.eval()

    all_preds = []
    all_labels = []
    all_overall = []

    for batch in tqdm(loader, desc="Predict"):
        batch = move_batch_to_device(batch)

        preds = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            lengths=batch["lengths"],
        )

        all_preds.append(preds.detach().cpu().numpy())
        all_labels.append(batch["labels"].detach().cpu().numpy())
        all_overall.append(batch["overall_band"].detach().cpu().numpy())

    return (
        np.concatenate(all_labels, axis=0),
        np.concatenate(all_preds, axis=0),
        np.concatenate(all_overall, axis=0),
    )

def print_metrics(metrics, title):
    print("\n" + title)
    print("=" * len(title))

    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")

def evaluate_lstm(model, loader, name="val"):
    y_true, y_pred, overall_true = predict_loader_lstm(model, loader)

    metrics, y_true_band, y_pred_band, overall_pred = compute_metrics(
        y_true,
        y_pred,
        overall_true,
    )

    print_metrics(metrics, f"== {name} ==")

    return metrics, y_true_band, y_pred_band, overall_pred

In [12]:
# =========================
# TRAIN LSTM ESSAY-ONLY MODEL WITH EARLY STOPPING
# Main metric: mean_trait_qwk
# Also reports derived_overall_qwk if available
# =========================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)

epochs_no_improve = 0
best_val_qwk = -1.0

best_path = os.path.join(RNN_OUTPUT_DIR, "best_lstm_essay_only.pt")

for epoch in range(1, EPOCHS + 1):
    model.train()

    running_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")

    for step, batch in enumerate(pbar, start=1):
        batch = move_batch_to_device(batch)

        optimizer.zero_grad(set_to_none=True)

        preds = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            lengths=batch["lengths"],
        )

        loss = criterion_loss_lstm(
            preds,
            batch["labels"],
            batch["overall_band"],
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM,
        )

        optimizer.step()

        running_loss += loss.item()
        avg_running_loss = running_loss / step

        pbar.set_postfix({
            "loss": f"{avg_running_loss:.6f}",
            "lr": optimizer.param_groups[0]["lr"],
        })

    avg_train_loss = running_loss / len(train_loader)

    print(f"\nEpoch {epoch} train loss: {avg_train_loss:.6f}")

    # =========================
    # Validation
    # =========================

    val_metrics, _, _, _ = evaluate_lstm(
        model,
        val_loader,
        name=f"val epoch {epoch}",
    )

    # =========================
    # Choose validation metric
    # Main metric should be mean_trait_qwk for multi-trait scoring
    # =========================

    if "mean_trait_qwk" in val_metrics:
        current_qwk = val_metrics["mean_trait_qwk"]
        selected_metric_name = "mean_trait_qwk"

    elif "avg_trait_qwk" in val_metrics:
        current_qwk = val_metrics["avg_trait_qwk"]
        selected_metric_name = "avg_trait_qwk"

    elif "trait_mean_qwk" in val_metrics:
        current_qwk = val_metrics["trait_mean_qwk"]
        selected_metric_name = "trait_mean_qwk"

    else:
        qwk_keys = [
            k for k in val_metrics.keys()
            if "qwk" in k.lower()
            and "overall" not in k.lower()
        ]

        if len(qwk_keys) == 0:
            qwk_keys = [
                k for k in val_metrics.keys()
                if "qwk" in k.lower()
            ]

        if len(qwk_keys) == 0:
            raise ValueError(
                "No QWK metric found in val_metrics. "
                f"Available keys: {list(val_metrics.keys())}"
            )

        current_qwk = float(np.nanmean([val_metrics[k] for k in qwk_keys]))
        selected_metric_name = "mean_of_" + "_".join(qwk_keys)

    scheduler.step(current_qwk)

    print(f"Selected validation metric: {selected_metric_name}")
    print(f"Current val QWK: {current_qwk:.4f}")
    print(f"Best val QWK so far: {best_val_qwk:.4f}")

    if "derived_overall_qwk" in val_metrics:
        print(f"Val derived_overall_qwk: {val_metrics['derived_overall_qwk']:.4f}")

    if "derived_overall_mae" in val_metrics:
        print(f"Val derived_overall_mae: {val_metrics['derived_overall_mae']:.4f}")

    if "derived_overall_from_traits_qwk" in val_metrics:
        print(
            "Val derived_overall_from_traits_qwk: "
            f"{val_metrics['derived_overall_from_traits_qwk']:.4f}"
        )

    # =========================
    # Save best checkpoint + early stopping
    # =========================

    improved = current_qwk > best_val_qwk + MIN_DELTA

    if improved:
        best_val_qwk = current_qwk
        epochs_no_improve = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),

                "best_val_qwk": best_val_qwk,
                "selected_metric_name": selected_metric_name,
                "best_val_metrics": val_metrics,

                "vocab_stoi": vocab.stoi,
                "vocab_itos": vocab.itos,

                "config": {
                    "MAX_LEN": MAX_LEN,
                    "MAX_VOCAB_SIZE": MAX_VOCAB_SIZE,
                    "MIN_FREQ": MIN_FREQ,

                    "EMBED_DIM": EMBED_DIM,
                    "RNN_HIDDEN": RNN_HIDDEN,
                    "RNN_LAYERS": RNN_LAYERS,
                    "BIDIRECTIONAL": BIDIRECTIONAL,

                    "STACK_HIDDEN": STACK_HIDDEN,
                    "DROPOUT": DROPOUT,

                    "BATCH_SIZE": BATCH_SIZE,
                    "EPOCHS": EPOCHS,
                    "LR": LR,
                    "WEIGHT_DECAY": WEIGHT_DECAY,
                    "MAX_GRAD_NORM": MAX_GRAD_NORM,

                    "TRAIT_COLS": TRAIT_COLS,
                    "MIN_BAND": MIN_BAND,
                    "MAX_BAND": MAX_BAND,

                    "USE_OPTIONAL_OVERALL_CONSISTENCY_LOSS": USE_OPTIONAL_OVERALL_CONSISTENCY_LOSS,
                    "OVERALL_CONSISTENCY_WEIGHT": OVERALL_CONSISTENCY_WEIGHT,

                    "EARLY_STOPPING_PATIENCE": EARLY_STOPPING_PATIENCE,
                    "MIN_DELTA": MIN_DELTA,

                    "FEATURE_MODE": "essay_only",
                },
            },
            best_path,
        )

        print(f"[SAVE] Best model saved to: {best_path}")
        print(f"[SAVE] best_val_qwk = {best_val_qwk:.4f}")

        if "derived_overall_qwk" in val_metrics:
            print(
                "[SAVE] derived_overall_qwk = "
                f"{val_metrics['derived_overall_qwk']:.4f}"
            )

    else:
        epochs_no_improve += 1

        print(
            f"[EARLY STOPPING] No improvement for "
            f"{epochs_no_improve}/{EARLY_STOPPING_PATIENCE} epochs."
        )

        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print("[EARLY STOPPING] Triggered.")
            print(f"Stopped at epoch {epoch}.")
            break

    print("-" * 80)

print("\nTraining finished.")
print(f"Best validation QWK: {best_val_qwk:.4f}")
print(f"Best checkpoint path: {best_path}")

Epoch 1/80:   0%|          | 0/242 [00:00<?, ?it/s]


Epoch 1 train loss: 0.033101


Predict:   0%|          | 0/31 [00:00<?, ?it/s]


== val epoch 1 ==
TR_qwk: 0.1110
TR_mae: 1.0929
CC_qwk: 0.1405
CC_mae: 1.2188
LR_qwk: 0.1582
LR_mae: 1.1502
GRA_qwk: 0.1807
GRA_mae: 1.2090
mean_trait_qwk: 0.1476
mean_trait_mae: 1.1677
derived_overall_from_traits_qwk: 0.1437
derived_overall_from_traits_mae: 1.1517
Selected validation metric: mean_trait_qwk
Current val QWK: 0.1476
Best val QWK so far: -1.0000
Val derived_overall_from_traits_qwk: 0.1437
[SAVE] Best model saved to: /content/ielts_lstm_essay_only_trait_stacks/best_lstm_essay_only.pt
[SAVE] best_val_qwk = 0.1476
--------------------------------------------------------------------------------


Epoch 2/80:   0%|          | 0/242 [00:00<?, ?it/s]

...


== val epoch 19 ==
TR_qwk: 0.3673
TR_mae: 1.1445
CC_qwk: 0.3558
CC_mae: 1.2812
LR_qwk: 0.3915
LR_mae: 1.2141
GRA_qwk: 0.3993
GRA_mae: 1.2554
mean_trait_qwk: 0.3785
mean_trait_mae: 1.2238
derived_overall_from_traits_qwk: 0.3808
derived_overall_from_traits_mae: 1.2172
Selected validation metric: mean_trait_qwk
Current val QWK: 0.3785
Best val QWK so far: 0.3830
Val derived_overall_from_traits_qwk: 0.3808
[EARLY STOPPING] No improvement for 7/8 epochs.
--------------------------------------------------------------------------------


Epoch 20/80:   0%|          | 0/242 [00:00<?, ?it/s]


Epoch 20 train loss: 0.004455


Predict:   0%|          | 0/31 [00:00<?, ?it/s]


== val epoch 20 ==
TR_qwk: 0.3685
TR_mae: 1.1574
CC_qwk: 0.3469
CC_mae: 1.2941
LR_qwk: 0.3884
LR_mae: 1.2234
GRA_qwk: 0.4054
GRA_mae: 1.2554
mean_trait_qwk: 0.3773
mean_trait_mae: 1.2326
derived_overall_from_traits_qwk: 0.3758
derived_overall_from_traits_mae: 1.2250
Selected validation metric: mean_trait_qwk
Current val QWK: 0.3773
Best val QWK so far: 0.3830
Val derived_overall_from_traits_qwk: 0.3758
[EARLY STOPPING] No improvement for 8/8 epochs.
[EARLY STOPPING] Triggered.
Stopped at epoch 20.

Training finished.
Best validation QWK: 0.3830
Best checkpoint path: /content/ielts_lstm_essay_only_trait_stacks/best_lstm_essay_only.pt


In [14]:
# =========================
# LOAD BEST CHECKPOINT AND EVALUATE ON TEST SET
# =========================

checkpoint = torch.load(
    best_path,
    map_location=DEVICE,
    weights_only=False,   # PyTorch 2.6+ fix
)

# Load model weights
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
model.eval()

print("Loaded checkpoint from:", best_path)
print("Best epoch:", checkpoint.get("epoch"))
print("Best validation QWK:", checkpoint.get("best_val_qwk"))
print("Selected metric:", checkpoint.get("selected_metric_name"))

test_metrics, y_true_band, y_pred_band, overall_pred = evaluate_lstm(
    model,
    test_loader,
    name="test best LSTM essay+discourse",
)

Loaded checkpoint from: /content/ielts_lstm_essay_only_trait_stacks/best_lstm_essay_only.pt
Best epoch: 12
Best validation QWK: 0.3829973788912625
Selected metric: mean_trait_qwk


Predict:   0%|          | 0/31 [00:00<?, ?it/s]


== test best LSTM essay+discourse ==
TR_qwk: 0.3364
TR_mae: 1.2263
CC_qwk: 0.3523
CC_mae: 1.3320
LR_qwk: 0.3817
LR_mae: 1.2552
GRA_qwk: 0.3980
GRA_mae: 1.3072
mean_trait_qwk: 0.3671
mean_trait_mae: 1.2802
derived_overall_from_traits_qwk: 0.3690
derived_overall_from_traits_mae: 1.2799


In [15]:
def build_prediction_df_lstm(df, y_true_band, y_pred_band, overall_pred):
    out = df.reset_index(drop=True).copy()

    for i, trait in enumerate(TRAIT_COLS):
        out[f"gold_{trait}"] = y_true_band[:, i]
        out[f"pred_{trait}"] = y_pred_band[:, i]
        out[f"abs_error_{trait}"] = (
            out[f"pred_{trait}"] - out[f"gold_{trait}"]
        ).abs()

    out["pred_Overall_derived"] = overall_pred

    if "Overall" in out.columns:
        out["gold_Overall"] = out["Overall"]

        valid = ~pd.isna(out["gold_Overall"])

        if valid.any():
            print("Derived Overall QWK:",
                  qwk_band(out.loc[valid, "gold_Overall"],
                           out.loc[valid, "pred_Overall_derived"]))

            print("Derived Overall MAE:",
                  mean_absolute_error(out.loc[valid, "gold_Overall"],
                                      out.loc[valid, "pred_Overall_derived"]))

    out["mean_abs_trait_error"] = out[
        [f"abs_error_{t}" for t in TRAIT_COLS]
    ].mean(axis=1)

    return out

pred_df_lstm = build_prediction_df_lstm(
    test_df,
    y_true_band,
    y_pred_band,
    overall_pred,
)

pred_path = os.path.join(
    RNN_OUTPUT_DIR,
    "test_predictions_lstm_essay_only_trait_stacks.csv",
)

pred_df_lstm.to_csv(pred_path, index=False)

print("Saved:", pred_path)
display(pred_df_lstm.head())

Saved: /content/ielts_lstm_essay_only_trait_stacks/test_predictions_lstm_essay_only_trait_stacks.csv


,prompt,essay,evaluation,essay_len,TR,CC,LR,GRA,n_found,parse_quality,...,pred_CC,abs_error_CC,gold_LR,pred_LR,abs_error_LR,gold_GRA,pred_GRA,abs_error_GRA,pred_Overall_derived,mean_abs_trait_error
0,18.Education of young people is highly priorit...,It can be argued that the vast majority of cou...,Task Achievement: \r\nThe essay adequately add...,303,7.0,7.0,6.0,6.0,4,good,...,7.0,0.0,6.0,6.5,0.5,6.0,6.5,0.5,7.0,0.250
1,Nowadays in many countries most shops and prod...,"In today's modern world, daily use things are ...",Task Achievement:\r\nThe candidate has adequat...,300,6.0,5.5,5.5,5.5,4,good,...,6.0,0.5,5.5,6.0,0.5,5.5,6.0,0.5,6.0,0.375
2,Some people believe that the rage of technolog...,Technology is deeply relative to the living of...,Task Achievement: 5.5\r\n- The candidate has e...,296,5.5,6.0,6.0,6.0,4,good,...,6.5,0.5,6.0,6.5,0.5,6.0,6.0,0.0,6.5,0.500
3,The world has many towns and cites constructed...,Numerous cities have houses and replanning and...,Task Achievement:\r\n\r\nThe essay adequately ...,342,6.5,6.5,6.0,6.0,4,good,...,7.0,0.5,6.0,7.0,1.0,6.0,6.5,0.5,7.0,0.750
4,Many people argue that in order to improve the...,"In this modern world, the importance of higher...",Task Achievement: (Band Score: 4)\r\n- The can...,264,4.0,3.5,3.0,3.0,4,good,...,5.0,1.5,3.0,5.0,2.0,3.0,5.0,2.0,5.0,1.750


In [16]:
summary_rows = []

for trait in TRAIT_COLS:
    qwk = qwk_band(
        pred_df_lstm[f"gold_{trait}"],
        pred_df_lstm[f"pred_{trait}"],
    )

    mae = mean_absolute_error(
        pred_df_lstm[f"gold_{trait}"],
        pred_df_lstm[f"pred_{trait}"],
    )

    summary_rows.append({
        "model": "BiLSTM essay-only",
        "target": trait,
        "QWK": qwk,
        "MAE": mae,
    })

if "gold_Overall" in pred_df_lstm.columns:
    valid = ~pd.isna(pred_df_lstm["gold_Overall"])

    if valid.any():
        qwk = qwk_band(
            pred_df_lstm.loc[valid, "gold_Overall"],
            pred_df_lstm.loc[valid, "pred_Overall_derived"],
        )

        mae = mean_absolute_error(
            pred_df_lstm.loc[valid, "gold_Overall"],
            pred_df_lstm.loc[valid, "pred_Overall_derived"],
        )

        summary_rows.append({
            "model": "BiLSTM essay-only",
            "target": "Overall_derived",
            "QWK": qwk,
            "MAE": mae,
        })

summary_df_lstm = pd.DataFrame(summary_rows)

summary_path = os.path.join(
    RNN_OUTPUT_DIR,
    "test_metric_summary_lstm_essay_only.csv",
)

summary_df_lstm.to_csv(summary_path, index=False)

print("Saved:", summary_path)
display(summary_df_lstm)

Saved: /content/ielts_lstm_essay_only_trait_stacks/test_metric_summary_lstm_essay_only.csv


,model,target,QWK,MAE
0,BiLSTM essay-only,TR,0.336374,1.226289
1,BiLSTM essay-only,CC,0.352329,1.331959
2,BiLSTM essay-only,LR,0.381684,1.255155
3,BiLSTM essay-only,GRA,0.398000,1.307217
